# Discount integrity — analysis walkthrough

This notebook is the readable version of what `pf analyse all` does. It exists so a
reader can follow the argument without reading the package.

**Run `make demo` first** if you don't have live data yet — it builds a synthetic
fixture so every cell below executes.

> Synthetic rows carry `source='synthetic'` and are excluded from any reported
> figure. The `INCLUDE_SYNTHETIC` switch below exists for development only.

In [ ]:
import sys, sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / "src"))

from priceforensics.analysis import inflation, mrp_audit, dark_patterns

INCLUDE_SYNTHETIC = True   # set False once live data has accumulated

DB = Path.cwd().parent / "data" / "prices.db"
conn = sqlite3.connect(DB)

pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. What have we actually collected?

Before any analysis: how much data is there, over what period, and are there gaps?
A longitudinal study with an unnoticed two-week hole produces confident nonsense.

In [ ]:
coverage = pd.read_sql_query("""
    SELECT d.date, l.site_key,
           COUNT(*) AS observations,
           COUNT(DISTINCT o.listing_id) AS listings
    FROM fact_price_observation o
    JOIN dim_date d    ON d.date_key   = o.date_key
    JOIN dim_listing l ON l.listing_id = o.listing_id
    GROUP BY d.date, l.site_key
    ORDER BY d.date
""", conn, parse_dates=["date"])

print(f"{coverage.observations.sum():,} observations")
print(f"{coverage.date.min():%Y-%m-%d} to {coverage.date.max():%Y-%m-%d}")
print(f"{coverage.date.nunique()} distinct collection days")

coverage.pivot_table(index="date", columns="site_key", values="observations",
                     aggfunc="sum").plot(title="Daily collection volume by site")
plt.ylabel("observations"); plt.xlabel(""); plt.show()

## 2. The advertised discount distribution

What retailers *claim*. This is the number a shopper sees, and it is the thing we
will spend the rest of the notebook testing.

In [ ]:
claims = pd.read_sql_query("""
    SELECT p.category, l.site_key, o.computed_discount_pct AS discount_pct
    FROM fact_price_observation o
    JOIN dim_listing l ON l.listing_id = o.listing_id
    JOIN dim_product p ON p.product_id = o.product_id
    WHERE o.computed_discount_pct IS NOT NULL
""", conn)

display(claims.groupby("category").discount_pct.describe()[["count", "mean", "50%", "max"]].round(1))

claims.discount_pct.plot(kind="hist", bins=40, edgecolor="white",
                         title="Advertised discount % — all listings")
plt.xlabel("advertised discount (%)"); plt.show()

## 3. Do sellers even agree on the MRP?

MRP is manufacturer-declared and printed on the pack — a property of the *product*,
not the *shop*. So the same SKU should carry the same MRP everywhere.

Where it doesn't, at least one seller is quoting a number that isn't the MRP. Since
that number is the denominator of the advertised discount, an inflated MRP
manufactures a saving that never existed.

**This needs one day of data.**

In [ ]:
findings = mrp_audit.find_contradictions(include_synthetic=INCLUDE_SYNTHETIC)
summary  = mrp_audit.summarise(findings)

print(f"{summary.get('n_contradictions', 0)} products where sellers disagree on MRP")
print(f"median spread: {summary.get('median_spread_pct')}%   worst: {summary.get('max_spread_pct')}%\n")

for f in findings[:6]:
    quotes = "  ".join(f"{site}=₹{mrp:,.0f}" for site, mrp in f.quotes)
    print(f"{f.canonical_title[:34]:<34} {quotes}")

## 4. Was the "before" price real?

The longitudinal test. We look for the signature of a manufactured discount: a price
rise held for several days, then reversed exactly when the sale starts.

Thresholds come from `config/targets.yaml` and were committed **before** collection
began — see `git log`. A detector tuned after seeing results can produce any
headline you like.

In [ ]:
events = inflation.detect(include_backfill=INCLUDE_SYNTHETIC)
n_listings = pd.read_sql_query(
    "SELECT COUNT(DISTINCT listing_id) n FROM fact_price_observation", conn).n[0]

summary = inflation.summarise(events, n_listings)
for k, v in summary.items():
    if k != "worst_example":
        print(f"{k:<28} {v}")
print(f"\nworst case:\n  {summary.get('worst_example')}")

### Claimed vs real discount

The whole project in one chart. Each bar pair is one flagged listing: what the site
advertised, against what a shopper actually saved versus the pre-rise price.

In [ ]:
if events:
    df = pd.DataFrame([{
        "title": e.title[:26],
        "claimed": e.claimed_discount_pct,
        "real": e.real_discount_pct,
        "overstated_pp": e.overstatement_pp,
        "site": e.site_key,
        "confidence": e.confidence,
    } for e in events]).sort_values("overstated_pp", ascending=False)

    top = df.head(12).set_index("title")
    top[["claimed", "real"]].plot(kind="barh", color=["#c0392b", "#27ae60"])
    plt.title("Advertised discount vs discount against the pre-rise price")
    plt.xlabel("discount (%)"); plt.ylabel(""); plt.gca().invert_yaxis()
    plt.legend(["advertised", "actual"]); plt.tight_layout(); plt.show()

    display(df.head(10).round(1))
else:
    print("No events yet — needs more price history.")

### A single product's story

The chart that makes the argument without any explanation needed.

In [ ]:
if events:
    e = events[0]
    series = pd.read_sql_query("""
        SELECT d.date, o.selling_price
        FROM fact_price_observation o
        JOIN dim_date d ON d.date_key = o.date_key
        WHERE o.listing_id = ? ORDER BY d.date
    """, conn, params=(e.listing_id,), parse_dates=["date"])

    ax = series.plot(x="date", y="selling_price", legend=False, lw=2, color="#2c3e50")
    ax.axhline(e.baseline_price, ls="--", color="#27ae60",
               label=f"real baseline ₹{e.baseline_price:,.0f}")
    ax.axhline(e.peak_price, ls="--", color="#c0392b",
               label=f"'before' price ₹{e.peak_price:,.0f}")
    ax.axvspan(pd.Timestamp(e.rise_start), pd.Timestamp(e.sale_start),
               alpha=0.12, color="red", label="inflation window")
    ax.set_title(f"{e.title} — {e.site_key}\n"
                 f"advertised {e.claimed_discount_pct:.0f}% off · "
                 f"real saving {e.real_discount_pct:.0f}% · "
                 f"overstated by {e.overstatement_pp:.0f}pp")
    ax.set_ylabel("₹"); ax.set_xlabel(""); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 5. Is "Only 3 left!" true?

A genuine stock counter moves. One that reports the same number every day for weeks,
while the item stays purchasable, is decoration.

In [ ]:
claims = dark_patterns.analyse()
summary = dark_patterns.summarise(claims)

if summary.get("n_assessed"):
    print(f"{summary['pct_static']}% of stock counters never changed "
          f"(n={summary['n_assessed']})")
    print(f"longest unchanged claim: {summary['longest_static_run_days']} days\n")
    print(summary.get("worst_example", ""))

    by_site = pd.DataFrame(summary["by_site"]).T
    (by_site["static"] / by_site["assessed"] * 100).plot(
        kind="bar", color="#e67e22", title="% of stock counters that never move")
    plt.ylabel("%"); plt.xticks(rotation=0); plt.show()
else:
    print("Not enough observation days yet.")

## 6. How much should you trust the detector?

Two different questions, two different numbers.

**Precision — are the flags real?** Measured by hand-reviewing a random sample and
reporting a Wilson score interval. Run `pf validate sample --run N` to draw one.

**Recall — what did it miss?** Unanswerable on real data, so it is measured against
planted synthetic events.

The recall number below is a **floor, not a ceiling**: the generator plants exactly
the pattern the detector searches for, so a perfect score confirms the implementation
matches its specification — not that it survives real, messy retail data. Manual
review is the number that actually matters.

In [ ]:
from priceforensics import synthetic

try:
    display(pd.Series(synthetic.evaluate_detector()).to_frame("value"))
except FileNotFoundError:
    print("No synthetic ground truth — run `pf synth generate` first.")

conn.close()